# MINI Cells Experiment 012 — Shared-Rule Settling Dynamics

Tests the Level-2 hypothesis: a language NCA should learn an autonomous shared update rule whose state and predictions settle when the same rule is free-run beyond the trained depth.

New training: 1D settling and 2D K=4 settling, 2M tokens each. Baselines are the already-published Experiment 011 stable checkpoints. This Kaggle launcher requires T4 x2 so the paired new models run on separate GPUs.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

ROOT = Path('/kaggle/working/mini-cells')
REF = 'codex/experiment-012-settling-dynamics'
os.chdir('/kaggle/working')
if ROOT.exists():
    shutil.rmtree(ROOT)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REF, 'https://github.com/ArcheLabs/mini-cells.git', str(ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lm,dev]'], cwd=ROOT, check=True)
os.chdir(ROOT)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)

In [ ]:
import torch
gpu_count = torch.cuda.device_count()
print({'python': sys.version.split()[0], 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu_count': gpu_count, 'gpus': [torch.cuda.get_device_name(i) for i in range(gpu_count)]})
assert torch.cuda.is_available(), 'CUDA is required'
assert gpu_count >= 2, 'Experiment 012 Kaggle launcher requires GPU T4 x2'

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pytest',
    'tests/test_language_settling.py',
    'tests/test_language_stabilization.py',
    'tests/test_language_2d.py',
    'tests/test_language_halting.py',
    '-q',
], cwd=ROOT, check=True)

In [ ]:
subprocess.run([sys.executable, 'scripts/run_language_settling.py'], cwd=ROOT, check=True)

In [ ]:
import json, pandas as pd
from IPython.display import Image, display
OUT = ROOT / 'results' / 'language-settling-dynamics-v1'
decision = json.loads((OUT / 'decision.json').read_text(encoding='utf-8'))
summary = pd.read_csv(OUT / 'model-summary.csv')
sweep = pd.read_csv(OUT / 'relaxation-sweep.csv')
print(json.dumps(decision, indent=2))
display(summary)
display(sweep)

In [ ]:
for name in [
    'ppl-vs-relaxation-depth.png',
    'residual-vs-relaxation-depth.png',
    'late-ppl-drift.png',
    'residual-contraction.png',
    'quality-vs-relaxation-compute.png',
]:
    print(name)
    display(Image(filename=str(OUT / name)))

In [ ]:
# Review decision.json and plots first. Then set PUBLISH=True to persist curated results.
PUBLISH = False
if PUBLISH:
    subprocess.run([sys.executable, 'scripts/publish_experiment_012_results.py', '--push'], cwd=ROOT, check=True)